In [ ]:
import os
import requests
import zipfile
import io

# 1. Fijar la ruta de tu infraestructura
ruta_raw = r"C:\Users\hugod\costco-expansion-analysis\data\raw"
os.makedirs(ruta_raw, exist_ok=True) # Crea la carpeta si no existe

# 2. Vectores de ataque (Jalisco=14, Nuevo León=19, Querétaro=22)
# El servidor del INEGI tiene una estructura de URLs predecible para las descargas masivas
estados = ['14', '19', '22']
base_url = "https://www.inegi.org.mx/contenidos/masiva/denue/denue_{}_csv.zip"

print("Iniciando extracción automatizada de infraestructura de mercado (INEGI)...")

# 3. Bucle de descarga y descompresión en memoria
for estado in estados:
    url = base_url.format(estado)
    print(f"-> Conectando al nodo del estado {estado}...")
    
    # Hacer la petición HTTP
    response = requests.get(url)
    
    if response.status_code == 200:
        # Descomprimir directamente desde la memoria RAM a tu disco duro
        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
            # Extraemos todo el contenido
            zip_ref.extractall(ruta_raw)
        print(f"   [ÉXITO] Datos del estado {estado} extraídos en data/raw/")
    else:
        print(f"   [FALLO] No se pudo conectar. Código HTTP: {response.status_code}")

print("\nPipeline finalizado. Los medios de producción de datos están en tu disco duro.")

In [5]:
import pandas as pd
import numpy as np
import glob
import os

print("Iniciando fusión de Oferta (DENUE) y Demanda (ENIGH)...")

# --------------------------------------------------------
# 1. PROCESAR LA OFERTA Y COMPETENCIA (DENUE)
# --------------------------------------------------------
ruta_raw = r"C:\Users\hugod\costco-expansion-analysis\data\raw"
archivos_denue = glob.glob(os.path.join(ruta_raw, '**', 'denue_inegi_*.csv'), recursive=True)

lista_df = []
for archivo in archivos_denue:
    df_temp = pd.read_csv(archivo, encoding='latin1', low_memory=False)
    df_temp.columns = df_temp.columns.str.lower()
    lista_df.append(df_temp)

denue = pd.concat(lista_df, ignore_index=True)

denue['ubica_geo'] = denue['cve_ent'].astype(str).str.zfill(2) + denue['cve_mun'].astype(str).str.zfill(3)
denue['ubica_geo'] = denue['ubica_geo'].astype(int)

nom_comercial = denue['nom_estab'].str.upper().fillna('')
scian = denue['codigo_act'].astype(str)

denue['competencia_club'] = np.where(nom_comercial.str.contains("SAM'S|SAMS|CITY CLUB"), 1, 0)
denue['es_b2b'] = np.where(scian.str.startswith(('722', '462112', '461110')), 1, 0)

oferta_mun = denue.groupby('ubica_geo').agg(
    total_competidores=('competencia_club', 'sum'),
    masa_negocios_b2b=('es_b2b', 'sum')
).reset_index()

# --------------------------------------------------------
# 2. PROCESAR LA DEMANDA (ENIGH)
# --------------------------------------------------------
ruta_enigh = r"C:\Users\hugod\costco-expansion-analysis\data\processed\datos_modelo_final.csv"
enigh = pd.read_csv(ruta_enigh)
enigh = enigh.copy() # Evita el PerformanceWarning de Pandas

# Calcular ingreso de mercado real
enigh['ingreso_mercado'] = enigh['ing_cor'] * enigh['factor']

# Verificamos si existe la variable de autos para no romper el código
tiene_autos = 'num_auto' in enigh.columns
if tiene_autos:
    enigh['autos_mercado'] = enigh['num_auto'] * enigh['factor']

# Agrupación dinámica
def agrupar_demanda(x):
    datos = {
        'poblacion_hogares': x['factor'].sum(),
        'ingreso_total_zona': x['ingreso_mercado'].sum()
    }
    if tiene_autos:
        datos['parque_vehicular'] = x['autos_mercado'].sum()
    return pd.Series(datos)

demanda_mun = enigh.groupby('ubica_geo').apply(agrupar_demanda).reset_index()
demanda_mun['ingreso_promedio_hogar'] = demanda_mun['ingreso_total_zona'] / demanda_mun['poblacion_hogares']

# --------------------------------------------------------
# 3. LA MATRIZ DE DECISIÓN (MERGE)
# --------------------------------------------------------
matriz_capital = pd.merge(demanda_mun, oferta_mun, on='ubica_geo', how='inner')
percentil_ingreso = matriz_capital['ingreso_promedio_hogar'].quantile(0.60)

matriz_capital['Y_viable'] = np.where(
    (matriz_capital['poblacion_hogares'] >= 150000) & 
    (matriz_capital['ingreso_promedio_hogar'] >= percentil_ingreso), 
    1, 0
)

ruta_salida = r"C:\Users\hugod\costco-expansion-analysis\data\processed\matriz_espacial_fase2.csv"
matriz_capital.to_csv(ruta_salida, index=False)

print("\n¡Matriz construida con éxito!")
print(f"Total de municipios analizados: {len(matriz_capital)}")
display(matriz_capital.sort_values(by='ingreso_total_zona', ascending=False).head(5))

Iniciando fusión de Oferta (DENUE) y Demanda (ENIGH)...

¡Matriz construida con éxito!
Total de municipios analizados: 98


,ubica_geo,poblacion_hogares,ingreso_total_zona,ingreso_promedio_hogar,total_competidores,masa_negocios_b2b,Y_viable
40,14120,378581.0,3.067339e+10,81022.006828,7,12547,1
16,14039,394804.0,2.896380e+10,73362.470173,7,18508,1
72,19039,272651.0,2.693443e+10,98787.194734,10,11865,1
94,22014,300834.0,2.626152e+10,87295.721557,10,12017,1
48,19006,292048.0,2.404342e+10,82326.955457,2,4642,1


In [6]:
"""
================================================================================
AUDITORÍA COMPLETA DEL PROYECTO COSTCO
Analiza todos los notebooks, datos y resultados del proyecto.
================================================================================
"""

import json
import pandas as pd
import numpy as np
from pathlib import Path
from nbformat import read

print("=" * 90)
print("AUDITORÍA COMPLETA: PROYECTO COSTCO MÉXICO")
print("=" * 90)

# 1. Listar todos los notebooks
notebooks_dir = Path(r'C:\Users\hugod\costco-expansion-analysis\notebooks')
notebooks = sorted(
    path for path in notebooks_dir.glob('*.ipynb')
    if not path.name.startswith('.')
)

print(f"\nNOTEBOOKS ENCONTRADOS: {len(notebooks)}\n")
for number, notebook_path in enumerate(notebooks, 1):
    print(f"{number}. {notebook_path.name}")

# 2. Analizar cada notebook
auditorias = {}
for notebook_path in notebooks:
    print("\n" + "=" * 90)
    print(f"ANALIZANDO: {notebook_path.name}")
    print("=" * 90)

    try:
        with open(notebook_path, encoding='utf-8') as file:
            notebook = read(file, as_version=4)

        n_cells = len(notebook.cells)
        n_markdown = sum(cell.cell_type == 'markdown' for cell in notebook.cells)
        n_code = sum(cell.cell_type == 'code' for cell in notebook.cells)
        full_code = '\n'.join(
            cell.source for cell in notebook.cells if cell.cell_type == 'code'
        )

        print(f"\nESTRUCTURA:\n   Total de celdas: {n_cells}\n   - Markdown: {n_markdown}\n   - Código: {n_code}")

        imports = set()
        import_rules = {
            'pandas': ('import pandas', 'import pd'),
            'numpy': ('import numpy', 'import np'),
            'statsmodels': ('import statsmodels', 'import sm'),
            'matplotlib/seaborn': ('import matplotlib', 'import seaborn'),
            'sklearn': ('import sklearn', 'from sklearn'),
        }
        for library, rules in import_rules.items():
            if any(rule in full_code for rule in rules):
                imports.add(library)

        data_files = set()
        output_files = set()
        for line in full_code.splitlines():
            if 'read_csv' in line and ('/' in line or '\\' in line):
                filename = line.split('\\')[-1].split('/')[-1].strip(" '\"),")
                if filename:
                    data_files.add(filename)
            if any(operation in line for operation in ('to_csv', 'savefig', 'modelo_resumen')):
                filename = line.split('\\')[-1].split('/')[-1].strip(" '\"),")
                if filename and '.' in filename:
                    output_files.add(filename)

        analyses = {
            'OLS': 'Regresión OLS',
            'logit': 'Regresión logística',
            'describe()': 'Estadísticas descriptivas',
            'hist': 'Histogramas',
            'scatter': 'Gráficos scatter',
            'boxplot': 'Box plots',
            'merge': 'Unión de tablas',
            'groupby': 'Agregaciones por grupos',
        }
        detected_analyses = [
            description for keyword, description in analyses.items()
            if keyword.lower() in full_code.lower()
        ]

        print(f"   Librerías: {', '.join(sorted(imports)) or 'ninguna'}")
        print(f"   Datos detectados: {', '.join(sorted(data_files)) or 'ninguno'}")
        print(f"   Salidas detectadas: {', '.join(sorted(output_files)) or 'ninguna'}")
        print(f"   Análisis: {', '.join(detected_analyses) or 'ninguno'}")

        auditorias[notebook_path.name] = {
            'celdas_totales': n_cells,
            'celdas_markdown': n_markdown,
            'celdas_codigo': n_code,
            'librerias': sorted(imports),
            'datos_entrada': sorted(data_files),
            'datos_salida': sorted(output_files),
            'analisis': detected_analyses,
            'ruta': str(notebook_path),
        }
    except Exception as error:
        print(f"   ERROR al analizar: {error}")

# 3. Resumen global de datos procesados y resultados
print("\n" + "=" * 90)
print("RESUMEN GLOBAL DEL PROYECTO")
print("=" * 90)

data_dir = Path(r'C:\Users\hugod\costco-expansion-analysis\data\processed')
print("\nARCHIVOS DE DATOS PROCESADOS:")
for csv_path in sorted(data_dir.glob('*.csv')):
    data = pd.read_csv(csv_path, low_memory=False, encoding='utf-8-sig')
    print(f"   {csv_path.name}: {len(data):,} filas | {len(data.columns)} columnas")

results_dir = Path(r'C:\Users\hugod\costco-expansion-analysis\resultados')
print("\nRESULTADOS GENERADOS:")
for result_path in sorted(path for path in results_dir.rglob('*') if path.is_file()):
    print(f"   {result_path.name}: {result_path.stat().st_size / 1024:.1f} KB")

print("\n" + "=" * 90)
print("QUÉ SE LOGRÓ HASTA AHORA")
print("=" * 90)
print("""
COMPLETADO:
   1. Descarga y extracción de datos ENIGH 2022.
   2. Filtrado para Jalisco, Nuevo León y Querétaro.
   3. Limpieza y agregación de ingresos por hogar.
   4. Unión de tablas de hogares e ingresos.
   5. Modelo econométrico OLS inicial.
   6. Gráficos exploratorios.

PENDIENTE PARA SAR-LOGIT:
   1. Integrar datos DENUE.
   2. Agregar variables macroeconómicas.
   3. Crear la matriz de pesos espaciales W.
   4. Cambiar la unidad de análisis de hogares a municipios.
   5. Definir una variable dependiente binaria de viabilidad.
   6. Estimar modelos espaciales SAR y SEM.
""")

# 4. Estadísticas clave, con conversión numérica segura
modelo_path = data_dir / 'datos_modelo_final.csv'
if modelo_path.exists():
    df_modelo = pd.read_csv(modelo_path, low_memory=False, encoding='utf-8-sig')
    df_modelo.columns = df_modelo.columns.str.replace('ï»¿', '', regex=False)
    for column in ('ingreso_total', 'est_socio', 'tam_loc'):
        if column in df_modelo.columns:
            df_modelo[column] = pd.to_numeric(df_modelo[column], errors='coerce')

    print("\nESTADÍSTICAS CLAVE:")
    print(f"   Hogares válidos: {len(df_modelo):,}")
    print(f"   Ingreso promedio: ${df_modelo['ingreso_total'].mean():,.0f} MXN")
    print(f"   Ingreso mediano: ${df_modelo['ingreso_total'].median():,.0f} MXN")
    print(f"   Estrato promedio: {df_modelo['est_socio'].mean():.2f}")
    print(f"   Tamaño de localidad promedio: {df_modelo['tam_loc'].mean():.2f}")

print("\n" + "=" * 90)
print("PRÓXIMOS PASOS PARA SAR-LOGIT")
print("=" * 90)
print("Integrar DENUE, datos macroeconómicos, matriz espacial W y una variable")
print("binaria de viabilidad a nivel municipal.")

# 5. Exportar auditoría a JSON
output_path = results_dir / 'AUDITORIA_PROYECTO.json'
with open(output_path, 'w', encoding='utf-8') as file:
    json.dump(auditorias, file, indent=2, ensure_ascii=False)

print(f"\nAUDITORÍA COMPLETA GUARDADA EN: {output_path}")

AUDITORÍA COMPLETA: PROYECTO COSTCO MÉXICO

NOTEBOOKS ENCONTRADOS: 7

1. 01_exploracion_datos.ipynb
2. 01_exploracion_datos_BACKUP.ipynb
3. 02_modelo_econometrico_final.ipynb
4. 03_extraccion_denue.ipynb
5. 03b_inyectar_vehiculos.ipynb
6. 04_modelo_logit.ipynb
7. Untitled-1.ipynb

ANALIZANDO: 01_exploracion_datos.ipynb

ESTRUCTURA:
   Total de celdas: 11
   - Markdown: 1
   - Código: 10
   Librerías: matplotlib/seaborn, numpy, pandas, sklearn, statsmodels
   Datos detectados: conjunto_de_datos_ingresos_enigh2022_ns.csv', encoding='utf-8-sig, datos_modelo.csv', encoding='latin-1, datos_modelo.csv', encoding='utf-8-sig, enigh_monterrey_guadalajara_queretaro.csv, enigh_monterrey_guadalajara_queretaro.csv', encoding='utf-8-sig, ingresos_por_hogar.csv
   Salidas detectadas: datos_modelo.csv', index=False, df_model.to_csv(, fig.savefig(, ingresos_por_hogar.csv', index=False, modelo_resumen.txt
   Análisis: Regresión OLS, Estadísticas descriptivas, Histogramas, Gráficos scatter, Box plots, Un